# Boltz2 Batch Notebook v1.0

Run this notebook top-to-bottom. It is organized as **7 main steps** for easy batch prediction with Boltz2.

## Workflow
1. Environment setup + workspace bootstrap
2. Helper module build (internal functions)
3. Input + manifest builder (CSV / YAML ZIP / YAML folder)
4. Validation + preflight + run profile setup
5. Launch or resume batch run
6. Summarize and rank results
7. Visualize structures and inspect logs
8. Export + download all run files

## What input can you provide?
You can use either:
- **CSV mode**: one row = one job
- **YAML ZIP mode**: zip of multiple `.yaml` files
- **YAML folder mode**: already uploaded YAML files

## CSV Example (recommended start)
Required columns:
- `job_name`
- `protein_sequence`

Optional columns:
- `ligand_smiles` or `ligand_ccd` (choose one)
- `protein_id`, `ligand_id`, `binder`
- `msa_mode` (`server` or `none`), `msa_path`
- `template_file`, `template_chain_id`, `template_id`
- `notes`

### Example cases
- **Protein only**: no ligand fields
- **Protein + small molecule**: set `ligand_smiles` and `binder` to ligand chain id (for example `B`)
- **Template-guided**: provide `template_file` path (`.pdb`, `.cif`, `.mmcif`)

## Tips
- Start with `RUN_PROFILE = "fast"` for pipeline checks.
- Use `RUN_PROFILE = "scientific"` for better-quality production screening.
- Final cell can export and download **all run artifacts as one ZIP**.

In [ ]:
# @title Step 1 - Environment Setup + Workspace Bootstrap
# Installs dependencies and prepares workspace folders. Example: run once at the start of a fresh Colab session.
import os, sys, subprocess, shutil, platform, json
from pathlib import Path

#@markdown ### Step 1 Help
#@markdown Use this step to prepare the runtime and optionally connect Google Drive.
#@markdown - **MOUNT_DRIVE**: `True` mounts Drive so you can load CSVs from Drive and persist exports.
#@markdown - **DRIVE_PROJECT_DIR**: Base Drive folder used by this notebook.
#@markdown - **AUTO_EXPORT_TO_DRIVE**: If enabled, Step 8 automatically copies the final ZIP to Drive.
MOUNT_DRIVE = True  # @param {type:"boolean"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/boltz_batch"  # @param {type:"string"}
AUTO_EXPORT_TO_DRIVE = True  # @param {type:"boolean"}

WORK_ROOT = Path("/content/boltz_batch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir("/content")

def run(cmd, quiet=False, check=True):
    print("$", " ".join(map(str, cmd)))
    kwargs = {}
    if quiet:
        kwargs["stdout"] = subprocess.DEVNULL
        kwargs["stderr"] = subprocess.DEVNULL
    return subprocess.run(cmd, check=check, text=True, **kwargs)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

drive_mounted = False
drive_paths = {
    "project_dir": "",
    "inputs_dir": "",
    "exports_dir": "",
}

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_mounted = Path("/content/drive/MyDrive").exists()
    except Exception as e:
        print("Drive mount skipped/unavailable:", e)

if drive_mounted:
    drive_project = Path(DRIVE_PROJECT_DIR)
    drive_inputs = drive_project / "inputs"
    drive_exports = drive_project / "exports"
    drive_inputs.mkdir(parents=True, exist_ok=True)
    drive_exports.mkdir(parents=True, exist_ok=True)

    drive_paths = {
        "project_dir": str(drive_project),
        "inputs_dir": str(drive_inputs),
        "exports_dir": str(drive_exports),
    }

    print("Drive connected:", drive_project)
    print("Drive inputs folder:", drive_inputs)

BOLTZ_REPO = os.environ.get("BOLTZ_REPO", "https://github.com/AtharvaTilewale/boltz.git")
if not Path("/content/boltz").exists():
    run(["git", "clone", BOLTZ_REPO])
else:
    print("Using existing /content/boltz")

# Single place for all dependency installs.
run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/boltz[cuda]"])
run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.1.0", "scipy", "numba", "pyyaml", "pandas", "ipywidgets", "py3Dmol", "biopython"])

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch diagnostics skipped:", e)

WORK_ROOT = Path("/content/boltz_batch")
DIRS = {
    "inputs": WORK_ROOT / "inputs",
    "tables": WORK_ROOT / "inputs" / "tables",
    "yamls": WORK_ROOT / "inputs" / "yamls",
    "templates": WORK_ROOT / "inputs" / "templates",
    "msas": WORK_ROOT / "inputs" / "msas",
    "uploads": WORK_ROOT / "uploads",
    "jobs": WORK_ROOT / "jobs",
    "manifests": WORK_ROOT / "manifests",
    "summaries": WORK_ROOT / "summaries",
    "exports": WORK_ROOT / "exports",
    "logs": WORK_ROOT / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Download Batch_Utils.py from GitHub
print("\nDownloading Batch_Utils.py...")
import urllib.request
github_url = "https://raw.githubusercontent.com/AtharvaTilewale/boltz2-notebook/refs/heads/main/scripts/batch_utils.py"
batch_utils_path = WORK_ROOT / "Batch_Utils.py"
try:
    urllib.request.urlretrieve(github_url, str(batch_utils_path))
    print(f"Successfully downloaded Batch_Utils.py to {batch_utils_path}")
except Exception as e:
    print(f"Error downloading Batch_Utils.py: {e}")

state = {
    "work_root": str(WORK_ROOT),
    "dirs": {k: str(v) for k, v in DIRS.items()},
    "drive": {
        "mounted": bool(drive_mounted),
        "auto_export_to_drive": bool(AUTO_EXPORT_TO_DRIVE),
        **drive_paths,
    },
}
with open(WORK_ROOT / "workspace_state.json", "w") as fh:
    json.dump(state, fh, indent=2)

print("\nWorkspace ready at:", WORK_ROOT)
print("Boltz binary:", shutil.which("boltz"))
if state["drive"]["mounted"]:
    print("Drive exports will be copied to:", state["drive"]["exports_dir"])

In [ ]:
# @title Step 2 - Input Assistant + Upload + Manifest Builder
# Choose INPUT_MODE: csv, fasta, yaml_zip, or yaml_dir. Provide your own input files.
import pandas as pd, json, sys, pathlib
from pathlib import Path

# Ensure helper module is importable
utils_path = "/content/boltz_batch"
if utils_path not in sys.path:
    sys.path.append(utils_path)

import importlib
import batch_utils # Import the module to be reloaded
importlib.reload(batch_utils) # Reload it to pick up latest changes

from batch_utils import (
    discover_uploads, save_uploaded_file,
    extract_yaml_zip, build_manifest_from_csv, build_manifest_from_yaml_dir, build_manifest_from_fasta
  )

#@markdown ### Step 2 Help
#@markdown Provide input files and build the manifest used by later steps.
#@markdown - **UPLOAD_FILES**: `True` opens the uploader to place files into notebook folders.
#@markdown - **INPUT_MODE** options:
#@markdown   - `csv`: one table where each row is one job.
#@markdown   - `fasta`: protein-only FASTA input. Header text after `>` is used as job name.
#@markdown   - `yaml_zip`: a `.zip` containing multiple YAML job files.
#@markdown   - `yaml_dir`: a folder path containing YAML files already present in runtime storage.
#@markdown - **FASTA mode rules**: only protein chains are accepted; ligand/dna/rna are not added.
#@markdown - If FASTA sequence contains `:`, each segment is treated as a separate protein chain.
#@markdown - **CSV_PATH / FASTA_PATH / ZIP_PATH / YAML_DIR**: use the field that matches your selected mode.
#@markdown - When uploads are off and a path is provided, this step searches common runtime and Drive roots to resolve the path.
UPLOAD_FILES = False  # @param {type:"boolean"}
if UPLOAD_FILES:
    from google.colab import files
    print("Upload supported: CSV, FASTA, YAML ZIP, YAML, PDB/CIF, A3M")
    uploaded = files.upload()
    saved = []
    for name, content in uploaded.items():
        ext = pathlib.Path(name).suffix.lower()
        if ext == ".csv":
            path = save_uploaded_file(name, content, category="tables")
        elif ext in {".fa", ".fasta", ".faa"}:
            path = save_uploaded_file(name, content, category="tables")
        elif ext == ".zip":
            path = save_uploaded_file(name, content, category="uploads")
        elif ext in {".pdb", ".cif", ".mmcif"}:
            path = save_uploaded_file(name, content, category="templates")
        elif ext == ".a3m":
            path = save_uploaded_file(name, content, category="msas")
        elif ext in {".yaml", ".yml"}:
            path = save_uploaded_file(name, content, category="yamls")
        else:
            path = save_uploaded_file(name, content, category="uploads")
        saved.append(path)
    print("Saved files:")
    for s in saved:
        print(" -", s)

uploads = discover_uploads()
print("\nCurrent inventory:")
print(json.dumps(uploads, indent=2))

# ---- INPUT SETTINGS ----
INPUT_MODE = "fasta"   # @param ["csv", "fasta", "yaml_zip", "yaml_dir"]
CSV_PATH = ""  # @param {type:"string"}
FASTA_PATH = ""  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
YAML_DIR = ""  # @param {type:"string"}
MANIFEST_NAME = "batch_manifest_v1"  # @param {type:"string"}


def _candidate_roots() -> list[Path]:
    roots = [Path.cwd(), Path("/content"), Path("/content/boltz_batch")]
    state_path = Path("/content/boltz_batch/workspace_state.json")
    if state_path.exists():
        try:
            ws = json.loads(state_path.read_text())
            drive_cfg = ws.get("drive", {}) if isinstance(ws, dict) else {}
            for key in ["project_dir", "inputs_dir"]:
                v = str(drive_cfg.get(key, "")).strip()
                if v:
                    roots.append(Path(v))
        except Exception:
            pass
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        roots.append(mydrive)

    unique = []
    seen = set()
    for r in roots:
        rs = str(r)
        if rs not in seen and r.exists():
            unique.append(r)
            seen.add(rs)
    return unique


def _resolve_input_path(raw_path: str, expect: str = "file") -> Path:
    q = str(raw_path).strip()
    if not q:
        raise ValueError("Input path is empty")

    requested = Path(q).expanduser()
    direct_candidates = []
    if requested.is_absolute():
        direct_candidates.append(requested)
    else:
        for root in _candidate_roots():
            direct_candidates.append(root / requested)

    def _matches(p: Path) -> bool:
        if expect == "file":
            return p.is_file()
        if expect == "dir":
            return p.is_dir()
        return p.exists()

    for c in direct_candidates:
        if _matches(c):
            return c.resolve()

    name = requested.name
    if not name:
        raise FileNotFoundError(f"Could not resolve path: {q}")

    for root in _candidate_roots():
        try:
            for p in root.rglob(name):
                if _matches(p):
                    return p.resolve()
        except Exception:
            continue

    raise FileNotFoundError(f"Could not resolve path: {q}")


if INPUT_MODE == "csv":
    if not str(CSV_PATH).strip():
        raise ValueError("CSV_PATH is required when INPUT_MODE='csv'.")
    if not UPLOAD_FILES:
        CSV_PATH = str(_resolve_input_path(CSV_PATH, expect="file"))
        print("Resolved CSV_PATH:", CSV_PATH)
    manifest_path = build_manifest_from_csv(CSV_PATH, manifest_name=MANIFEST_NAME)
elif INPUT_MODE == "fasta":
    if not str(FASTA_PATH).strip():
        raise ValueError("FASTA_PATH is required when INPUT_MODE='fasta'.")
    if not UPLOAD_FILES:
        FASTA_PATH = str(_resolve_input_path(FASTA_PATH, expect="file"))
        print("Resolved FASTA_PATH:", FASTA_PATH)
    manifest_path = build_manifest_from_fasta(FASTA_PATH, manifest_name=MANIFEST_NAME)
elif INPUT_MODE == "yaml_zip":
    if not str(ZIP_PATH).strip():
        raise ValueError("ZIP_PATH is required when INPUT_MODE='yaml_zip'")
    if not UPLOAD_FILES:
        ZIP_PATH = str(_resolve_input_path(ZIP_PATH, expect="file"))
        print("Resolved ZIP_PATH:", ZIP_PATH)
    extracted = extract_yaml_zip(ZIP_PATH)
    print("Extracted YAML files:", len(extracted))
    if not extracted:
        raise ValueError("No YAML files were found in the provided ZIP file")
    root = str(Path(extracted[0]).parent)
    manifest_path = build_manifest_from_yaml_dir(root, manifest_name=MANIFEST_NAME)
elif INPUT_MODE == "yaml_dir":
    if not str(YAML_DIR).strip():
        raise ValueError("YAML_DIR is required when INPUT_MODE='yaml_dir'")
    if not UPLOAD_FILES:
        YAML_DIR = str(_resolve_input_path(YAML_DIR, expect="dir"))
        print("Resolved YAML_DIR:", YAML_DIR)
    manifest_path = build_manifest_from_yaml_dir(YAML_DIR, manifest_name=MANIFEST_NAME)
else:
    raise ValueError("Unsupported INPUT_MODE")

print("\nManifest created:", manifest_path)
manifest_df = pd.read_csv(manifest_path)
display(manifest_df.head(20))

In [ ]:
# @title Step 3 - Validate + Preflight + Run Profile Setup
# Example: use RUN_PROFILE='fast' for smoke tests and 'scientific' for higher quality production runs.
import json
from pathlib import Path
from batch_utils import validate_manifest, preflight_manifest, apply_run_profile

#@markdown ### Step 3 Help
#@markdown Configure run parameters and validate jobs before launching.
#@markdown - **RUN_PROFILE** options:
#@markdown   - `fast`: quick sanity checks.
#@markdown   - `balanced`: default trade-off between speed and quality.
#@markdown   - `scientific`: heavier settings for quality-focused runs.
#@markdown - **OUTPUT_FORMAT** options:
#@markdown   - `pdb`: widely compatible default.
#@markdown   - `mmcif` / `cif`: structure output in CIF-based formats.
#@markdown - **SKIP_COMPLETED** skips already successful jobs; **RETRY_FAILED** re-runs failed jobs.
MANIFEST_PATH = "/content/boltz_batch/manifests/batch_manifest_v1.csv"  # @param {type:"string"}
RUN_PROFILE = "balanced"  # @param ["fast", "balanced", "scientific"]

# --- Form parameters (must use static defaults) ---
RECYCLING_STEPS = 3  # @param {type:"integer"}
SAMPLING_STEPS = 233  # @param {type:"integer"}
DIFFUSION_SAMPLES = 1  # @param {type:"integer"}
USE_POTENTIALS = False  # @param {type:"boolean"}
OUTPUT_FORMAT = "pdb"  # @param ["pdb", "mmcif", "cif"]
SKIP_COMPLETED = True  # @param {type:"boolean"}
RETRY_FAILED = False  # @param {type:"boolean"}

if not Path(MANIFEST_PATH).exists():
    print(f"Error: Manifest not found at {MANIFEST_PATH}. Please run Step 3 first.")
else:
    # Initialize with profile defaults
    RUN_PARAMS = apply_run_profile(RUN_PROFILE)

    # Override with manual form values
    RUN_PARAMS.update({
        "recycling_steps": max(1, int(RECYCLING_STEPS)),
        "sampling_steps": max(1, int(SAMPLING_STEPS)),
        "diffusion_samples": max(1, int(DIFFUSION_SAMPLES)),
        "use_potentials": bool(USE_POTENTIALS),
        "output_format": str(OUTPUT_FORMAT).lower(),
        "skip_completed": bool(SKIP_COMPLETED),
        "retry_failed": bool(RETRY_FAILED)
    })

    # Save parameters for the run engine
    with open("/content/boltz_batch/run_params_batch.json", "w") as fh:
        json.dump(RUN_PARAMS, fh, indent=2)

    validation_df = validate_manifest(MANIFEST_PATH)
    preflight_df = preflight_manifest(MANIFEST_PATH, RUN_PARAMS)

    print("Run profile:", RUN_PROFILE)
    print("\nValidation report:")
    display(validation_df)
    print("\nPreflight report:")
    display(preflight_df)

    print("\nSummary")
    print("Total jobs            :", len(validation_df))
    print("Validation issues     :", int((validation_df["issue_count"] > 0).sum()))
    print("Preflight hard issues :", int((preflight_df["issue_count"] > 0).sum()))
    print("Preflight warnings    :", int((preflight_df["warning_count"] > 0).sum()))
    print("Ready to run          :", int((preflight_df["ready_to_run"] == True).sum()))

In [ ]:
# @title Step 4 - Launch or Resume Batch Run
# Set RUN_MODE='launch' for first pass or RUN_MODE='resume' to continue interrupted jobs.
import json
import importlib
import batch_utils

# Always reload to pick up latest Step 2 edits (including realtime progress logging).
batch_utils = importlib.reload(batch_utils)
run_manifest = batch_utils.run_manifest
manifest_overview = batch_utils.manifest_overview
progress_table = batch_utils.progress_table

#@markdown ### Step 4 Help
#@markdown Start new jobs or continue an existing run.
#@markdown - **RUN_MODE** options:
#@markdown   - `launch`: starts jobs (optionally capped by MAX_JOBS_THIS_SESSION).
#@markdown   - `resume`: continues from manifest state and ignores MAX_JOBS_THIS_SESSION.
#@markdown - **VERBOSE_UPDATES**: when `True`, prints live updates (running job, completed, remaining).
MANIFEST_PATH = "/content/boltz_batch/manifests/batch_manifest_v1.csv"  # @param {type:"string"}
RUN_MODE = "launch"  # @param ["launch", "resume"]
MAX_JOBS_THIS_SESSION = 0  # @param {type:"integer"}
VERBOSE_UPDATES = True  # @param {type:"boolean"}

with open("/content/boltz_batch/run_params_batch.json") as fh:
    RUN_PARAMS = json.load(fh)

if RUN_MODE == "resume":
    # Resume ignores session cap and relies on skip_completed/retry_failed flags.
    max_jobs = None
else:
    max_jobs = None if MAX_JOBS_THIS_SESSION == 0 else int(MAX_JOBS_THIS_SESSION)

print("Using batch_utils from:", getattr(batch_utils, "__file__", "<unknown>"))
result_df = run_manifest(MANIFEST_PATH, run_params=RUN_PARAMS, max_jobs=max_jobs, verbose=bool(VERBOSE_UPDATES))
overview = manifest_overview(MANIFEST_PATH)
print("Overview:", overview)
display(progress_table(MANIFEST_PATH))

In [ ]:
# @title Step 5 - Summarize and Rank Results
# Example ranking: RANK_METRIC='affinity_probability_binary' for hit discovery.
import pandas as pd
from batch_utils import summarize_manifest, manifest_overview, rank_results

#@markdown ### Step 5 Help
#@markdown Summarize outputs and rank jobs by your selected metric.
#@markdown - **STATUS_FILTER** chooses which job states to include.
#@markdown - **RANK_METRIC** options:
#@markdown   - `auto`: picks the best available scoring column automatically.
#@markdown   - explicit metrics: force ranking by that column.
MANIFEST_PATH = "/content/boltz_batch/manifests/batch_manifest_v1.csv"  # @param {type:"string"}
TOP_N = 10  # @param {type:"integer"}
STATUS_FILTER = "COMPLETED"  # @param ["COMPLETED", "FAILED", "RUNNING", "PENDING", "ALL"]
RANK_METRIC = "auto"  # @param ["auto", "affinity_probability_binary", "affinity_pred_value", "confidence_score", "iptm", "complex_plddt", "ptm", "complex_pde"]

summary_df = summarize_manifest(MANIFEST_PATH)
print("Overview:", manifest_overview(MANIFEST_PATH))

priority_cols = [
    "job_name", "status", "rank_metric", "rank_direction",
    "confidence_score", "iptm", "complex_plddt",
    "affinity_pred_value", "affinity_probability_binary",
    "model_path", "runtime_sec"
]
show_cols = [c for c in priority_cols if c in summary_df.columns]
display(summary_df[show_cols])

ranked = rank_results(summary_df, top_n=TOP_N, status_filter=STATUS_FILTER, metric=RANK_METRIC)
print("\nTop ranked jobs:")
display(ranked)

summary_csv = "/content/boltz_batch/summaries/batch_results_summary.csv"
print("Summary written to:", summary_csv)

In [ ]:
# @title Step 6A - Visualize Completed Structures (Next Button)
# Leave JOB_NAME_TO_VIEW empty to start from the first completed model.
import pandas as pd
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

#@markdown ### Step 6A Help
#@markdown View generated structures interactively.
#@markdown - **SUMMARY_PATH**: path to summary CSV (created in Step 6).
#@markdown - **JOB_NAME_TO_VIEW**: optional specific job name; leave blank to start at the first completed job.
SUMMARY_PATH = "/content/boltz_batch/summaries/batch_results_summary.csv"  # @param {type:"string"}
JOB_NAME_TO_VIEW = ""  # @param {type:"string"}

summary_df = pd.read_csv(SUMMARY_PATH).fillna("")
completed_df = summary_df[summary_df["model_path"] != ""].reset_index(drop=True)

if completed_df.empty:
    raise ValueError("No completed models found in summary CSV.")

if JOB_NAME_TO_VIEW:
    matches = completed_df.index[completed_df["job_name"] == JOB_NAME_TO_VIEW].tolist()
    if not matches:
        raise ValueError(f"No completed model found for job_name={JOB_NAME_TO_VIEW!r}")
    current_idx = matches[0]
else:
    current_idx = 0

header = widgets.HTML()
prev_btn = widgets.Button(description="Previous", button_style="info")
next_btn = widgets.Button(description="Next", button_style="primary")
out = widgets.Output()

state = {"idx": current_idx}

def render_current_structure():
    row = completed_df.iloc[state["idx"]]
    job_name = str(row.get("job_name", ""))
    model_path = str(row.get("model_path", ""))
    model_format = str(row.get("model_format", "pdb") or "pdb").lower()

    header.value = (
        f"<b>Visualizing job:</b> {job_name} "
        f"<span style='color:#666'>( {state['idx'] + 1} / {len(completed_df)} )</span>"
    )

    with out:
        clear_output(wait=True)
        print("Job:", job_name)
        print("Model:", model_path)
        print("Format:", model_format)

        if not Path(model_path).exists():
            print("Model file not found on disk.")
            return

        with open(model_path) as fh:
            structure_block = fh.read()

        input_fmt = "pdb" if model_format == "pdb" else "cif"
        view = py3Dmol.view(width=900, height=650)
        view.addModel(structure_block, input_fmt)
        view.setStyle({"cartoon": {"color": "spectrum"}})
        view.zoomTo()
        view.show()

def on_next_clicked(_):
    state["idx"] = (state["idx"] + 1) % len(completed_df)
    render_current_structure()

def on_prev_clicked(_):
    state["idx"] = (state["idx"] - 1) % len(completed_df)
    render_current_structure()

prev_btn.on_click(on_prev_clicked)
next_btn.on_click(on_next_clicked)

display(widgets.HBox([prev_btn, next_btn]))
display(header)
display(out)

render_current_structure()

In [ ]:
# @title Step 6B - Inspect Job Logs and Failure Diagnostics
# Example: set JOB_NAME to inspect a specific failed run and view suggested fixes.
import pandas as pd
from pathlib import Path
from batch_utils import failure_diagnostics_table

#@markdown ### Step 6B Help
#@markdown Inspect logs and troubleshooting notes.
#@markdown - **MANIFEST_PATH**: manifest CSV to inspect.
#@markdown - **JOB_NAME**: leave blank to list jobs; set a value to show stdout/stderr and diagnostics for one job.
MANIFEST_PATH = "/content/boltz_batch/manifests/batch_manifest_v1.csv"  # @param {type:"string"}
JOB_NAME = ""  # @param {type:"string"}

manifest_df = pd.read_csv(MANIFEST_PATH).fillna("")
diag_df = failure_diagnostics_table(MANIFEST_PATH)
if len(diag_df):
    print("Failure diagnostics:")
    display(diag_df)

if not JOB_NAME:
    print("Available jobs:")
    show_cols = [c for c in ["job_name", "status", "stdout_log", "stderr_log"] if c in manifest_df.columns]
    display(manifest_df[show_cols])
else:
    row = manifest_df[manifest_df["job_name"] == JOB_NAME]
    if row.empty:
        raise ValueError("Unknown JOB_NAME")
    rec = row.iloc[0]
    print("Status:", rec.get("status", ""))

    status_file = Path(rec["status_file"])
    if status_file.exists():
        import json
        status = json.loads(status_file.read_text())
        if isinstance(status.get("diagnostic"), dict):
            print("\nLikely cause:", status["diagnostic"].get("likely_cause", ""))
            print("Suggested fix:", status["diagnostic"].get("suggested_fix", ""))

    print("\n--- STDOUT ---")
    sp = Path(rec["stdout_log"])
    print(sp.read_text()[:12000] if sp.exists() else "[missing]")
    print("\n--- STDERR ---")
    ep = Path(rec["stderr_log"])
    print(ep.read_text()[:12000] if ep.exists() else "[missing]")

In [ ]:
# @title Step 7 - Export and Download All Run Files (Single ZIP)
# Creates a single archive with predictions, logs, manifests, and summaries.
import json
import shutil
from pathlib import Path
from batch_utils import create_export_zip

#@markdown ### Step 7 Help
#@markdown Package all artifacts and optionally download.
#@markdown - **EXPORT_NAME**: output ZIP file name prefix.
#@markdown - **AUTO_DOWNLOAD**: if enabled, triggers browser download (Colab environments).
#@markdown - If Drive auto-export was enabled in Step 1, this step also copies ZIP to Drive exports folder.
MANIFEST_PATH = "/content/boltz_batch/manifests/batch_manifest_v1.csv"  # @param {type:"string"}
EXPORT_NAME = "boltz2_batch_all_runs_v1_0_1"  # @param {type:"string"}
AUTO_DOWNLOAD = True  # @param {type:"boolean"}

zip_path = create_export_zip(MANIFEST_PATH, export_name=EXPORT_NAME)
print("ZIP created:", zip_path)

# If Drive was configured in Step 1, copy export ZIP there automatically.
state_path = Path("/content/boltz_batch/workspace_state.json")
if state_path.exists():
    try:
        ws = json.loads(state_path.read_text())
        drive_cfg = ws.get("drive", {}) if isinstance(ws, dict) else {}
        if drive_cfg.get("mounted") and drive_cfg.get("auto_export_to_drive"):
            drive_exports_dir = str(drive_cfg.get("exports_dir", "")).strip()
            if drive_exports_dir:
                drive_exports = Path(drive_exports_dir)
                drive_exports.mkdir(parents=True, exist_ok=True)
                drive_zip_path = drive_exports / Path(zip_path).name
                shutil.copy2(zip_path, drive_zip_path)
                print("Copied ZIP to Drive:", drive_zip_path)
    except Exception as e:
        print("Drive export copy skipped:", e)

if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(Path(zip_path)))
    except Exception as e:
        print("Auto-download unavailable in this environment.")
        print("Manual ZIP path:", zip_path)
        print("Details:", e)